# RunPod Generation — Phase 0-2 (Module A only)

Generates PRIMERA summaries for the 100-case full-scale run. GPU-bound only —
Module B/C (CourtListener verification, correction loop) runs **locally**
afterward over several days (rate-limited to 125 requests/day), not here.

**Before running:** upload `code_bundle.zip` (built locally, see
`docs/RUNPOD_SETUP.md`) into the same folder as this notebook via the Jupyter
file browser.

| Cell | Does |
|---|---|
| 1 | Unzip code, set up paths |
| 2 | Confirm GPU is visible |
| 3 | Install dependencies |
| 4 | Set `HF_TOKEN` (typed via masked prompt, not saved to this file) |
| 5 | Pre-flight smoke checks |
| 6 | Load the 100 selected cases |
| 7 | **Generate** (resume-safe — re-running skips completed case_ids) |
| 8 | Wrap-up: what to download, remember to terminate the pod |


## 1. Unzip code + set up paths

In [1]:
import zipfile
from pathlib import Path

HERE = Path.cwd()
ZIP_PATH = HERE / "code_bundle.zip"
if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"{ZIP_PATH} not found. Upload code_bundle.zip into this notebook's folder first."
    )

if not (HERE / "code").exists():
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(HERE)
    print("Unzipped to", HERE / "code")
else:
    print("code/ already present, skipping unzip.")

WORKDIR = HERE / "code"
import os, sys
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR / "src"))
print("cwd:", Path.cwd())
print("src/ exists:", (WORKDIR / "src").exists())

code/ already present, skipping unzip.
cwd: /workspace/code/notebooks/code
src/ exists: True


## 2. Confirm GPU

In [2]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No CUDA device visible — check the pod's GPU template before continuing.")

Fri Jul 17 12:06:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:06:00.0 Off |                  Off |
|  0%   27C    P8             17W /  450W |       1MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Install dependencies

Numpy pinned `<2.0` — see `requirements-colab.txt` header for why.

In [3]:
import subprocess

req = WORKDIR / "requirements-colab.txt"
subprocess.run(["pip", "install", "-q", "-r", str(req)], check=True)

import numpy as np, datasets, pandas, transformers
print("numpy:", np.__version__)
print("datasets:", datasets.__version__)
if np.__version__.startswith("2."):
    raise RuntimeError(f"numpy 2.x may break pyarrow/datasets (got {np.__version__}). Re-run this cell.")
if not datasets.__version__.startswith(("2.", "3.")):
    raise RuntimeError(f"Expected datasets<4, got {datasets.__version__}.")
print("Dependency stack OK.")

numpy: 1.26.4
datasets: 3.6.0
Dependency stack OK.


## 4. Set HF_TOKEN

Typed via a masked prompt so it never gets written into this notebook file.
Only `HF_TOKEN` is needed here — `COURTLISTENER_API_TOKEN` stays on your
local machine, since Module C runs there.

In [4]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")
os.environ["HF_HOME"] = str(WORKDIR / "data" / "hf")
os.environ["MULTILEXSUM_CACHE"] = str(WORKDIR / "data" / "hf")
print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))

HF_TOKEN:  ········


HF_TOKEN set: True


## 5. Pre-flight smoke checks

In [5]:
from smoke.colab_smoke import smoke_environment, smoke_hf_token_only, smoke_multilexsum, smoke_module_b

for result in (
    smoke_environment(require_cuda=True),
    smoke_hf_token_only(),  # not smoke_secrets() -- this pod never holds COURTLISTENER_API_TOKEN
    smoke_multilexsum(),
    smoke_module_b(),
):
    status = "PASS" if result.passed else "FAIL"
    print(f"[{status}] {result.name} — {result.message}")
    if not result.passed:
        raise RuntimeError(f"Smoke check failed: {result.name} — {result.message}")

print("\nAll smoke checks passed.")

README.md:   0%|          | 0.00/7.00k [00:00<?, ?B/s]

multi_lexsum.py:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

releases/v20230518/train.json: reconstructing file:   0%|          |  0.00B / 17.6MB            

releases/v20230518/train.json: downloading bytes:           |  0.00B            

releases/v20230518/dev.json: reconstructing file:   0%|          |  0.00B / 2.56MB            

releases/v20230518/dev.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

[PASS] environment — numpy=1.26.4, datasets=3.6.0, cuda=True
[PASS] secrets — HF_TOKEN present
[PASS] multilexsum — 454 validation cases
[PASS] module_b — extracted 2 citations

All smoke checks passed.


## 6. Load the 100 selected cases (test split, D→L)

In [6]:
from transformers import AutoTokenizer

from utils.config import load_yaml
from data.load_multilexsum import load_multilexsum, clear_dataset_cache
from data.preprocessing import build_case_input, GRANULARITY_CHECKPOINTS
from data.case_selection import select_cases

EXPERIMENT_CONFIG = "experiment_test100.yaml"

exp_cfg = load_yaml(EXPERIMENT_CONFIG)
data_cfg = load_yaml("data.yaml")
mod_a = load_yaml("module_a.yaml")

SPLIT = exp_cfg["split"]
GRANULARITY = exp_cfg["granularity"]
MAX_CASES = exp_cfg["max_cases"]
assert GRANULARITY == "long", "This notebook is scoped to D→L (long) only."
print(f"Experiment: {EXPERIMENT_CONFIG} | {SPLIT}/{GRANULARITY} | max_cases={MAX_CASES}")

clear_dataset_cache()
ds = load_multilexsum(cache_dir=WORKDIR / "data" / "hf")
split = ds[SPLIT]
print(f"Split {SPLIT!r}: {len(split)} cases total")

tokenizer = AutoTokenizer.from_pretrained(GRANULARITY_CHECKPOINTS[GRANULARITY])
max_tokens = data_cfg["max_tokens"][GRANULARITY]

selected_rows = select_cases(
    split,
    case_ids=exp_cfg.get("case_ids"),
    max_cases=MAX_CASES,
    min_gold_citations=exp_cfg.get("min_gold_citations", 1),
    seed=exp_cfg.get("seed", 42),
)
missing = set(exp_cfg.get("case_ids") or []) - {r["id"] for r in selected_rows}
if missing:
    print(f"WARNING: {len(missing)} pinned case_ids not found: {sorted(missing)[:10]}")

cases = []
for row in selected_rows:
    case = build_case_input(row, GRANULARITY, tokenizer, max_tokens)
    if case is not None:
        cases.append(case)

print(f"Prepared {len(cases)}/{len(selected_rows)} CaseInput objects")
print(f"Truncated to {max_tokens} tokens: {sum(1 for c in cases if c.truncated)}/{len(cases)}")
assert len(cases) > 0, "No cases selected — check case_ids in the experiment YAML."
print("First case:", cases[0].case_id)

Experiment: experiment_test100.yaml | test/long | max_cases=100
Split 'test': 908 cases total


config.json:   0%|          | 0.00/1.90k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

Prepared 100/100 CaseInput objects
Truncated to 4096 tokens: 100/100
First case: SD-MI-0001


## 7. Generate (resume-safe)

Checkpoints to `outputs/generations_test100/test_long.jsonl` after every case.
If this cell is interrupted (kernel restart, pod hiccup), just **re-run it** —
already-written `case_id`s are skipped, so nothing already done is repeated.

Expect roughly 1-3 hours for 100 D→L cases on an RTX 4090 (beam=5, ~850 output
tokens/case). Progress prints per case via the tqdm bar below.

In [7]:
import gc
from module_a_summarizer import Summarizer, generate_batch, load_generations

mod_a_run = {
    **mod_a,
    "granularity": GRANULARITY,
    "device": "cuda",
    "use_fp16": True,
    "output_dir": str(WORKDIR / exp_cfg.get("generations_dir", "outputs/generations")),
}

summarizer = Summarizer(granularity=GRANULARITY, device="cuda", use_fp16=True)
gen_path = generate_batch(cases, mod_a_run, split=SPLIT, summarizer=summarizer)
generations = load_generations(gen_path)
print(f"\nGenerations: {len(generations)}/{len(cases)} cases → {gen_path}")

del summarizer
gc.collect()
torch.cuda.empty_cache()
print("PRIMERA unloaded.")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.79GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/263 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
ModuleA test/long:   0%|          | 0/100 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.79GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Both `max_new_tokens` (=850) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=600) and `min_length`(=128) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
ModuleA test/long:   1%|          | 1/100 [00:08<13:21,  8.09s/it][transformers] Both `max_new_tokens` (=850) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=600) and `min_length`(=128) seem to have been set. `min_new_tokens` will take precedence. Please r


Generations: 100/100 cases → /workspace/code/notebooks/code/outputs/generations_test100/test_long.jsonl
PRIMERA unloaded.


## 8. Done — next steps

In [8]:
print(f'''
Generation complete: {len(generations)}/{len(cases)} cases.

NEXT STEPS:
1. In the Jupyter file browser, right-click {gen_path.relative_to(WORKDIR)} -> Download.
2. Save it locally to: code/{gen_path.relative_to(WORKDIR)}
   (create the outputs/generations_test100/ folder locally if it doesn't exist yet)
3. Terminate this RunPod pod now -- generation is the only phase that needs the GPU.
4. Locally: python scripts/run_verification_local.py --config experiment_test100.yaml
   (Module B/C + correction + evaluation, CourtListener-rate-limited, runs over
   the next few days -- no GPU needed for this part.)
''')


Generation complete: 100/100 cases.

NEXT STEPS:
1. In the Jupyter file browser, right-click outputs/generations_test100/test_long.jsonl -> Download.
2. Save it locally to: code/outputs/generations_test100/test_long.jsonl
   (create the outputs/generations_test100/ folder locally if it doesn't exist yet)
3. Terminate this RunPod pod now -- generation is the only phase that needs the GPU.
4. Locally: python scripts/run_verification_local.py --config experiment_test100.yaml
   (Module B/C + correction + evaluation, CourtListener-rate-limited, runs over
   the next few days -- no GPU needed for this part.)

